In [5]:
import pandas as pd
import numpy as np

In [6]:
sales = pd.read_csv("sales.csv")
inventory = pd.read_csv("inventory.csv")
customers = pd.read_csv("customer.csv")

In [7]:
print("Sales Info")
print(sales.info())

Sales Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     15 non-null     int64  
 1   customer_id  15 non-null     object 
 2   product      15 non-null     object 
 3   category     15 non-null     object 
 4   price        14 non-null     float64
 5   quantity     14 non-null     float64
 6   date         14 non-null     object 
 7   city         15 non-null     object 
 8   channel      15 non-null     object 
dtypes: float64(2), int64(1), object(6)
memory usage: 1.2+ KB
None


In [8]:
print("Inventory Info")
print(inventory.info())

Inventory Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product        11 non-null     object
 1   stock          11 non-null     int64 
 2   warehouse      11 non-null     object
 3   reorder_level  11 non-null     int64 
 4   supplier       11 non-null     object
dtypes: int64(2), object(3)
memory usage: 572.0+ bytes
None


In [9]:
print("Customers Info")
print(customers.info())

Customers Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    15 non-null     object
 1   customer_name  15 non-null     object
 2   gender         15 non-null     object
 3   age            15 non-null     int64 
 4   city           15 non-null     object
 5   membership     15 non-null     object
dtypes: int64(1), object(5)
memory usage: 852.0+ bytes
None


In [10]:
sales.columns = sales.columns.str.lower().str.strip().str.replace(" ", "_")
inventory.columns = inventory.columns.str.lower().str.strip().str.replace(" ", "_")
customers.columns = customers.columns.str.lower().str.strip().str.replace(" ", "_")

In [11]:
sales["price"] = sales["price"].fillna(sales["price"].median())

In [12]:
sales["quantity"] = sales["quantity"].fillna(sales["quantity"].median())

In [13]:
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")

In [14]:
sales = sales.dropna(subset=["date"])

In [15]:
sales = sales.drop_duplicates()

In [16]:
sales = sales[(sales["price"] > 0) & (sales["quantity"] > 0)]

In [17]:
sales.to_csv("cleaned_sales.csv", index=False)

In [42]:
print("Week 1 Completed ✅")

Week 1 Completed ✅


In [18]:
merged = pd.merge(sales, inventory, on="product", how="left")

In [19]:
if "customer_id" in sales.columns and "customer_id" in customers.columns:
    merged = pd.merge(merged, customers, on="customer_id", how="left")

In [20]:
merged["total_sales"] = merged["price"] * merged["quantity"]

In [21]:
merged["order_day"] = merged["date"].dt.day
merged["month"] = merged["date"].dt.month
merged["weekday"] = merged["date"].dt.day_name()

In [22]:
merged["day_type"] = np.where(
    merged["weekday"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday")

In [24]:
if "stock" in merged.columns:
    merged["stock_status"] = np.where(
        merged["stock"] < 20, "Low",
        np.where(merged["stock"] <= 100, "Normal", "Excess"))

In [26]:
print("Missing Values:")
print(merged.isnull().sum())

Missing Values:
order_id         0
customer_id      0
product          0
category         0
price            0
quantity         0
date             0
city_x           0
channel          0
stock            0
warehouse        0
reorder_level    0
supplier         0
customer_name    0
gender           0
age              0
city_y           0
membership       0
total_sales      0
order_day        0
month            0
weekday          0
day_type         0
stock_status     0
dtype: int64


In [27]:
print("Duplicate Rows:", merged.duplicated().sum())

Duplicate Rows: 0


In [28]:
merged.to_csv("merged_dataset.csv", index=False)

In [50]:
print("Week 2 Completed ✅")

Week 2 Completed ✅


In [63]:
df = pd.read_csv("merged_dataset.csv")

In [62]:
Q1 = merged["price"].quantile(0.25)
Q3 = merged["price"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

merged = merged[(merged["price"] >= lower) & (merged["price"] <= upper)]

In [52]:
if "city" in merged.columns:
    merged["city"] = merged["city"].str.title().str.strip()

In [53]:
if "category" in merged.columns:
    merged["category"] = merged["category"].str.title().str.strip()

In [54]:
merged.to_csv("validated_dataset.csv", index=False)

In [55]:
print("Week 3 Completed ✅")

Week 3 Completed ✅


In [66]:
df.drop_duplicates(inplace=True)
df["date"] = pd.to_datetime(df["date"])

In [67]:
print("Final Dataset Shape:", merged.shape)
print("Null Values:")
print(merged.isnull().sum())

Final Dataset Shape: (3, 24)
Null Values:
order_id         0
customer_id      0
product          0
category         0
price            0
quantity         0
date             0
city_x           0
channel          0
stock            0
warehouse        0
reorder_level    0
supplier         0
customer_name    0
gender           0
age              0
city_y           0
membership       0
total_sales      0
order_day        0
month            0
weekday          0
day_type         0
stock_status     0
dtype: int64


In [57]:
print("Duplicate Rows:", merged.duplicated().sum())
print(merged.describe())

Duplicate Rows: 0
       order_id        price  quantity                 date      stock  \
count      6.00     6.000000  6.000000                    6   6.000000   
mean    1006.00  1300.000000  2.000000  2024-07-26 16:00:00  42.166667   
min     1001.00   800.000000  1.000000  2024-05-01 00:00:00  18.000000   
25%     1003.25  1200.000000  2.000000  2024-05-16 06:00:00  31.250000   
50%     1005.50  1200.000000  2.000000  2024-07-01 00:00:00  50.000000   
75%     1009.25  1350.000000  2.000000  2024-10-01 06:00:00  50.000000   
max     1011.00  2000.000000  3.000000  2024-12-01 00:00:00  60.000000   
std        4.00   394.968353  0.632456                  NaN  16.618263   

       reorder_level        age  total_sales  order_day      month  
count       6.000000   6.000000     6.000000        6.0   6.000000  
mean       17.500000  25.333333  2500.000000        1.0   7.833333  
min        10.000000  22.000000  1400.000000        1.0   5.000000  
25%        12.500000  24.000000  2400.0

In [68]:
merged.to_csv("final_dataset.csv", index=False)

In [69]:
print("Week 4 Completed ✅")

Week 4 Completed ✅


In [70]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.dtypes)
print((df["price"] <= 0).sum())
print((df["quantity"] <= 0).sum())

order_id         0
customer_id      0
product          0
category         0
price            0
quantity         0
date             0
city_x           0
channel          0
stock            0
warehouse        0
reorder_level    0
supplier         0
customer_name    0
gender           0
age              0
city_y           0
membership       0
total_sales      0
order_day        0
month            0
weekday          0
day_type         0
stock_status     0
dtype: int64
0
order_id                  int64
customer_id              object
product                  object
category                 object
price                   float64
quantity                float64
date             datetime64[ns]
city_x                   object
channel                  object
stock                     int64
warehouse                object
reorder_level             int64
supplier                 object
customer_name            object
gender                   object
age                       int64
city_y           

In [71]:
df.to_csv("final_dataset.csv", index=False)